Thanks to Jason Brownlee and his articles on transformers.

In [1]:
import pathlib

import tensorflow as tf

# download dataset provided by Anki: https://www.manythings.org/anki/
text_file = tf.keras.utils.get_file(
    fname="fra-eng.zip",
    origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
    extract=True,
)
# show where the file is located now
text_file = pathlib.Path(text_file).parent / "fra.txt"
print(text_file)

3423204/3423204 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
C:\Users\mamma\.keras\datasets\fra.txt


In [ ]:
text_file = r"C:\Users\mamma\.keras\datasets\fra-eng_extracted\fra.txt"
with open (text_file, "r") as handle:
    fra = handle.readlines()
fra = [line.split("\t") for line in fra]
fra =[[f[0], f[1].replace("\n","")] for f in fra]

In [22]:
import random
random.choice(fra)

["Are you sure you don't want to keep it?",
 'Êtes-vous sûres de ne pas vouloir la garder ?']

In [ ]:
class Lang:
    def __init__(self, language):
        self.language = language
        self.tokens = {}
        self.len = 0
        self.vocab_size = 1000

        self.seq_len = 20

        self.train_texts = None
        self.valid_texts = None

        self.vectorizer = None
        
en = Lang("en")
fr = Lang("fr")

In [58]:
en.len = [len(line.split()) for line,_ in fra]
fr.len = [len(line.split()) for _, line in fra]
max(en.len), max(fr.len)

(47, 54)

In [65]:
random.shuffle(fra)
n_val = int(.15 * len(fra))
n_train = len(fra) -2* n_val
train_pairs = fra[:n_train]
valid_pairs = fra[n_train:]


en.train_texts = [line for line, _ in train_pairs]
fr.train_texts = [line for _, line in train_pairs]
en.valid_texts = [line for line, _ in valid_pairs]
fr.valid_texts = [line for _, line in valid_pairs]


In [57]:
def my_tokenizer(tokens_en, eng_sentence):
    for sen in eng_sentence.split():
        if sen in tokens_en:
            tokens_en[sen]+=1
        else:
            tokens_en[sen] = 1

for eng_sentence , fr_sentence in fra:
    my_tokenizer(en.tokens, eng_sentence)
    my_tokenizer(fr.tokens, fr_sentence)
len(en.tokens), len(fr.tokens)

(27110, 44277)

In [70]:
en.seq_len

20

In [76]:
en.ds = tf.data.Dataset.from_tensor_slices(en.train_texts).batch(32)
fr.ds = tf.data.Dataset.from_tensor_slices(fr.train_texts).batch(32)


In [ ]:
# from above we set

from tensorflow.keras.layers import TextVectorization

# limit the the vectorizer to learn only the more frequent words and make the rare words as out-of-vocabulary (OOV).
# skip the words of little value or with spelling mistakes
en.vocab_size = 10000
fr.vocab_size = 20000


en.vectorizer = TextVectorization(
    max_tokens=en.vocab_size,
    standardize=None,
    split="whitespace",
    output_mode="int",
    output_sequence_length=en.seq_len,
)

fr.vectorizer = TextVectorization(
    max_tokens=fr.vocab_size,
    standardize=None,
    split="whitespace",
    output_mode="int",
    output_sequence_length=fr.seq_len,
)


In [74]:
print(type(fr.train_texts))


<class 'list'>


In [95]:
a = list(zip(*train_pairs[:3]))
a

[('What I can do for you?', 'Why did they hire you?', 'We both want it.'),
 ("Qu'est-ce que je peux faire pour toi ?",
  'Pourquoi vous ont-ils recrutée ?',
  'Nous le voulons tous les deux.')]

In [96]:
list(zip(*a))

[('What I can do for you?', "Qu'est-ce que je peux faire pour toi ?"),
 ('Why did they hire you?', 'Pourquoi vous ont-ils recrutée ?'),
 ('We both want it.', 'Nous le voulons tous les deux.')]

In [99]:
features = ["hello", "goodbye"]
labels = ["bonjour", "au revoir"]

ds = tf.data.Dataset.from_tensor_slices((features, labels))

for x, y in ds:
    print(x.numpy(), y.numpy())

b'hello' b'bonjour'
b'goodbye' b'au revoir'


In [ ]:
batch_sizse = 64
ds = tf.data.Dataset.from_tensor_slices((en.train_texts, fr.train_texts))
ds = ds.shuffle(2048).batch(batch_size=batch_sizse).map(lambda e,f: (en.vectorizer(e),fr.vectorizer(f)))

NameError: name 'f' is not defined

In [107]:
en.vectorizer(list(ds.take(2))[0])

<tf.Tensor: shape=(2, 20), dtype=int64, numpy=
array([[  45,    2,   51,   20,   15,   97,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0],
       [   1,    1,    1,    1,    1, 7894,    1,    1,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0]],
      dtype=int64)>

In [ ]:
en.vectorizer.adapt(en.train_texts)

fr.vectorizer.adapt(fr.train_texts)

TypeError: 'TextVectorization' object is not subscriptable

In [68]:
fr.vectorizer.get_config()

{'name': 'text_vectorization_1',
 'trainable': True,
 'dtype': {'module': 'keras',
  'class_name': 'DTypePolicy',
  'config': {'name': 'float32'},
  'registered_name': None},
 'max_tokens': 20000,
 'standardize': None,
 'split': 'whitespace',
 'ngrams': None,
 'output_mode': 'int',
 'output_sequence_length': 20,
 'pad_to_max_tokens': False,
 'sparse': False,
 'ragged': False,
 'vocabulary': None,
 'idf_weights': None,
 'encoding': 'utf-8',
 'vocabulary_size': 20000}